# Layer 2 — 正常訂單瓶頸拆解

排除 Layer 1a + 1b 異常訂單後，分析正常訂單的瓶頸在哪。
將 total_duration 拆成四段：queue, db, device, inner_processing。

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

REPORTS_DIR = Path('../reports')
PARALLELISM = 4

df = pd.read_csv('../data/orders.csv')
df['order_created_at'] = pd.to_datetime(df['order_created_at'], format='%Y/%m/%d %I:%M:%S %p')

# Load anomaly flags
sys_flags = pd.read_csv('../data/system_anomaly_flags.csv')
usr_flags = pd.read_csv('../data/user_anomaly_flags.csv')

df = df.merge(sys_flags, on='order_id')
df = df.merge(usr_flags[['order_id', 'is_user_anomaly']], on='order_id')

# Filter to normal orders only
df['is_anomaly'] = df['is_system_anomaly'] | df['is_user_anomaly']
normal = df[~df['is_anomaly']].copy()
print(f"Total: {len(df)}, Anomalies excluded: {df['is_anomaly'].sum()}, Normal: {len(normal)}")


Total: 30000, Anomalies excluded: 2967, Normal: 27033


## 1. Phase Duration 估算

每個 order 有 4 threads 並行處理，所以 order-level 耗時 ≈ avg × file_count / 4

In [2]:
# Estimate order-level phase durations
normal['est_queue'] = normal['queue_duration_seconds']
normal['est_db'] = normal['db_duration_avg_seconds'] * normal['file_count'] / PARALLELISM
normal['est_device'] = normal['device_duration_avg_seconds'] * normal['file_count'] / PARALLELISM
normal['est_inner'] = normal['inner_processing_duration_avg_seconds'] * normal['file_count'] / PARALLELISM
normal['est_total'] = normal['est_queue'] + normal['est_db'] + normal['est_device'] + normal['est_inner']

# File count groups
bins = [0, 50, 300, 1000, 2000, 5000]
labels = ['<50', '50-300', '300-1000', '1000-2000', '2000+']
normal['fc_group'] = pd.cut(normal['file_count'], bins=bins, labels=labels, right=True)

print("Orders per file_count group:")
print(normal['fc_group'].value_counts().sort_index())


Orders per file_count group:
fc_group
<50          8441
50-300       8287
300-1000     6994
1000-2000    3045
2000+         266
Name: count, dtype: int64


## 2. Phase 佔比分析

In [3]:
# Compute phase proportions per group
phase_cols = ['est_queue', 'est_db', 'est_device', 'est_inner']
phase_labels = ['Queue', 'DB', 'Device', 'Inner Processing']

group_means = normal.groupby('fc_group', observed=True)[phase_cols].mean()
group_means.columns = phase_labels

# Proportions
group_pct = group_means.div(group_means.sum(axis=1), axis=0) * 100

print("Phase proportion by file_count group (%):")
print(group_pct.round(1).to_string())
print()
print("Phase absolute means (seconds):")
print(group_means.round(1).to_string())


Phase proportion by file_count group (%):
           Queue    DB  Device  Inner Processing
fc_group                                        
<50         26.7  19.4    39.0              14.9
50-300       5.3  25.1    50.5              19.0
300-1000     1.5  26.2    52.4              19.9
1000-2000    0.6  26.3    53.1              19.9
2000+        0.4  27.2    52.9              19.5

Phase absolute means (seconds):
           Queue     DB  Device  Inner Processing
fc_group                                         
<50         14.4   10.5    21.0               8.1
50-300      14.2   66.9   134.4              50.5
300-1000    14.2  247.9   496.5             188.3
1000-2000   14.0  571.3  1150.9             432.1
2000+       14.4  939.9  1830.1             675.7


## 3. 圖表

In [4]:
# Chart 1: Phase proportion stacked bar (percentage)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#f39c12', '#3498db', '#e74c3c', '#2ecc71']

group_pct.plot(kind='bar', stacked=True, ax=axes[0], color=colors)
axes[0].set_title('Phase Proportion by File Count Group (%)')
axes[0].set_xlabel('File Count Group')
axes[0].set_ylabel('Proportion (%)')
axes[0].legend(title='Phase', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[0].tick_params(axis='x', rotation=0)

# Chart 2: Phase absolute stacked bar
group_means.plot(kind='bar', stacked=True, ax=axes[1], color=colors)
axes[1].set_title('Phase Duration by File Count Group (seconds)')
axes[1].set_xlabel('File Count Group')
axes[1].set_ylabel('Mean Duration (seconds)')
axes[1].legend(title='Phase', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(REPORTS_DIR / '2_phase_breakdown_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 2_phase_breakdown_bars.png")


Saved: 2_phase_breakdown_bars.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65643/3984567499.py:22: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [5]:
# Chart 3: Duration vs file_count scatter with trend
fig, ax = plt.subplots(figsize=(14, 6))

scatter = ax.scatter(normal['file_count'], normal['total_duration_seconds'],
                     alpha=0.1, s=5, c='steelblue')

# Add trend line
z = np.polyfit(normal['file_count'], normal['total_duration_seconds'], 1)
p = np.poly1d(z)
x_line = np.linspace(normal['file_count'].min(), normal['file_count'].max(), 100)
ax.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Trend: {z[0]:.2f}x + {z[1]:.0f}')

ax.set_title('Total Duration vs File Count (Normal Orders)')
ax.set_xlabel('File Count')
ax.set_ylabel('Total Duration (seconds)')
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / '2_duration_vs_filecount.png', dpi=150)
plt.show()
print("Saved: 2_duration_vs_filecount.png")


Saved: 2_duration_vs_filecount.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65643/1511222169.py:19: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [6]:
# Chart 4: Percentile analysis per group
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
phase_map = {'est_queue': 'Queue', 'est_db': 'DB', 'est_device': 'Device', 'est_inner': 'Inner Processing'}

for idx, (col, label) in enumerate(phase_map.items()):
    ax = axes[idx // 2][idx % 2]
    percentiles = normal.groupby('fc_group', observed=True)[col].quantile([0.5, 0.95, 0.99]).unstack()
    percentiles.columns = ['P50', 'P95', 'P99']
    percentiles.plot(kind='bar', ax=ax, color=['#2ecc71', '#f39c12', '#e74c3c'])
    ax.set_title(f'{label} Duration Percentiles by File Count Group')
    ax.set_xlabel('File Count Group')
    ax.set_ylabel('Duration (seconds)')
    ax.tick_params(axis='x', rotation=0)
    ax.legend()

plt.tight_layout()
plt.savefig(REPORTS_DIR / '2_phase_percentiles.png', dpi=150)
plt.show()
print("Saved: 2_phase_percentiles.png")


Saved: 2_phase_percentiles.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65643/2661363436.py:18: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 4. Summary

In [7]:
# Identify the biggest bottleneck per group
biggest = group_pct.idxmax(axis=1)
print("Biggest bottleneck per file_count group:")
for g, phase in biggest.items():
    print(f"  {g}: {phase} ({group_pct.loc[g, phase]:.1f}%)")

print(f"\n=== Layer 2 Summary ===")
print(f"Normal orders analyzed: {len(normal)}")
print(f"Overall dominant phase: {group_pct.mean().idxmax()} ({group_pct.mean().max():.1f}% avg)")


Biggest bottleneck per file_count group:
  <50: Device (39.0%)
  50-300: Device (50.5%)
  300-1000: Device (52.4%)
  1000-2000: Device (53.1%)
  2000+: Device (52.9%)

=== Layer 2 Summary ===
Normal orders analyzed: 27033
Overall dominant phase: Device (49.6% avg)
